# Laboratorio 09.2 — Fallos de agentes

Provocaremos errores de selección, argumentos inválidos, llamadas innecesarias, herramientas inventadas, ciclos y crecimiento de contexto usando decisores simulados.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Reproducir fallos frecuentes de un agente con entradas controladas.
- Distinguir error de selección de error de ejecución.
- Detectar llamadas repetidas y limitar intentos.
- Medir crecimiento del historial y diseñar salvaguardas.

## 2. Conceptos

Los agentes pueden escoger herramientas equivocadas, inventar nombres, enviar argumentos incompatibles, repetir acciones o continuar sin necesidad. Cada paso añade observaciones al contexto. Un bucle acotado, un catálogo cerrado, validación tipada, detección de repetición, presupuesto y registro de trazas hacen visibles estos modos de fallo.

## 3. Arquitectura

```mermaid
flowchart TD
    A[Decisor simulado] --> B[Validador Pydantic]
    B --> C{Tipo de fallo}
    C --> D[Tool equivocada]
    C --> E[Argumentos invalidos]
    C --> F[Herramienta inventada]
    C --> G[Llamada repetida]
    C --> H[Contexto creciente]
    D --> I[Traza y metricas]
    E --> I
    F --> I
    G --> I
    H --> I
```

## 4. Preparación del entorno

Reutilizamos el bucle manual y dispatcher de herramientas con respuestas simuladas. El laboratorio no invoca proveedores y cada escenario es reproducible.

In [ ]:
import json

import pandas as pd

from llm_engineering.environment import preparar_entorno
from llm_engineering.agents import run_agent

raiz_curso = preparar_entorno()

## 5. Definición del problema

Analizamos seis trazas que representan solicitudes de soporte. Para cada una registramos ruta esperada, herramientas utilizadas, errores y resultado terminal. Las expectativas sirven para ilustrar evaluación y deben acordarse con el producto real.

## 6. Línea base

Un agente mínimo sin restricciones podría llamar repetidamente herramientas o aceptar nombres inventados. Ejecutamos primero una secuencia correcta para tener una traza de referencia.

In [ ]:
def crear_decisor(decisiones: list[dict[str, object]]):
    """Devuelve respuestas JSON en orden, repitiendo la ultima al agotarlas."""
    indice = 0
    def decidir(solicitud: str, observaciones: list[dict[str, object]]) -> str:
        nonlocal indice
        decision = decisiones[min(indice, len(decisiones) - 1)]
        indice += 1
        return json.dumps(decision)
    return decidir

correcto = run_agent(
    'Calcula 6 * 7',
    crear_decisor([
        {'action': 'tool', 'tool_name': 'calculadora', 'arguments': {'expression': '6 * 7'}},
        {'action': 'final', 'final_answer': 'El resultado es 42.'},
    ]),
)
print(correcto.model_dump())

## 7. Implementación

Construimos casos de fallo deliberados. Las solicitudes y decisiones se registran por escenario para comparar qué detecta el runtime y qué requiere una etiqueta externa de herramienta esperada.

In [ ]:
escenarios = [
    {
        'caso': 'herramienta_equivocada', 'esperada': 'clima_mock', 'max_tool_calls': 2,
        'decisiones': [
            {'action': 'tool', 'tool_name': 'busqueda_productos', 'arguments': {'query': 'temperatura Santiago'}},
            {'action': 'final', 'final_answer': 'No encontré el dato.'},
        ],
    },
    {
        'caso': 'argumentos_invalidos', 'esperada': 'clima_mock', 'max_tool_calls': 2,
        'decisiones': [
            {'action': 'tool', 'tool_name': 'clima_mock', 'arguments': {'city': '', 'extra': 'valor'}},
            {'action': 'final', 'final_answer': 'No pude consultar el clima.'},
        ],
    },
    {
        'caso': 'llamada_innecesaria', 'esperada': None, 'max_tool_calls': 2,
        'decisiones': [
            {'action': 'tool', 'tool_name': 'clima_mock', 'arguments': {'city': 'Santiago'}},
            {'action': 'final', 'final_answer': 'Hola.'},
        ],
    },
    {
        'caso': 'bucle_repetido', 'esperada': 'calculadora', 'max_tool_calls': 4,
        'decisiones': [
            {'action': 'tool', 'tool_name': 'calculadora', 'arguments': {'expression': '1 + 1'}},
        ],
    },
    {
        'caso': 'herramienta_inventada', 'esperada': 'busqueda_documentos', 'max_tool_calls': 2,
        'decisiones': [
            {'action': 'tool', 'tool_name': 'buscar_en_internet', 'arguments': {'query': 'pedido'}},
        ],
    },
    {
        'caso': 'historial_creciente', 'esperada': 'clima_mock', 'max_tool_calls': 4,
        'decisiones': [
            {'action': 'tool', 'tool_name': 'clima_mock', 'arguments': {'city': 'Santiago'}},
            {'action': 'tool', 'tool_name': 'clima_mock', 'arguments': {'city': 'Valparaiso'}},
            {'action': 'tool', 'tool_name': 'clima_mock', 'arguments': {'city': 'Concepcion'}},
            {'action': 'final', 'final_answer': 'Terminé la consulta.'},
        ],
    },
]
filas_fallos = []
for escenario in escenarios:
    ejecucion = run_agent(
        escenario['caso'],
        crear_decisor(escenario['decisiones']),
        max_steps=6,
        max_tool_calls=escenario['max_tool_calls'],
    )
    llamadas = [paso for paso in ejecucion.steps if paso.get('action') == 'tool']
    herramienta_inicial = llamadas[0]['tool_name'] if llamadas else None
    seleccion_correcta = (
        herramienta_inicial is None if escenario['esperada'] is None
        else herramienta_inicial == escenario['esperada']
    )
    filas_fallos.append({
        'caso': escenario['caso'],
        'estado': ejecucion.status,
        'herramienta_esperada': escenario['esperada'],
        'herramienta_elegida': herramienta_inicial,
        'seleccion_correcta': seleccion_correcta,
        'llamadas': len(llamadas),
        'caracteres_observacion': sum(len(str(paso.get('output', ''))) for paso in llamadas),
        'fallos_ejecucion': sum(not paso.get('success', False) for paso in llamadas),
    })
tabla_fallos = pd.DataFrame(filas_fallos)
display(tabla_fallos)
tabla_fallos.to_csv(raiz_curso / 'outputs' / '09_02_agent_failures.csv', index=False)

## 8. Experimentos

Cambiamos el presupuesto de llamadas en el escenario de historial creciente. El decisor pide ciudades distintas para evitar la detección de una repetición exacta; el límite debe detenerlo aunque cada llamada sea diferente.

In [ ]:
decisiones_contexto = escenarios[-1]['decisiones']
filas_presupuesto = []
for presupuesto in (1, 2, 4):
    ejecucion = run_agent(
        'Compara clima de varias ciudades',
        crear_decisor(decisiones_contexto),
        max_steps=6,
        max_tool_calls=presupuesto,
    )
    filas_presupuesto.append({
        'presupuesto_herramientas': presupuesto,
        'estado': ejecucion.status,
        'llamadas': sum(paso.get('action') == 'tool' for paso in ejecucion.steps),
        'caracteres_historial': sum(len(str(paso.get('output', ''))) for paso in ejecucion.steps),
    })
tabla_presupuestos = pd.DataFrame(filas_presupuesto)
display(tabla_presupuestos)

## 9. Evaluación

Calculamos selección correcta, tasa de error de herramienta y fallo terminal. Una llamada innecesaria requiere comparar con la ruta esperada; no se detecta solo mirando si la herramienta devolvió éxito.

In [ ]:
metricas_fallos = pd.DataFrame([{
    'tool_selection_accuracy': float(tabla_fallos['seleccion_correcta'].mean()),
    'tool_execution_failure_rate': float(tabla_fallos['fallos_ejecucion'].sum() / max(tabla_fallos['llamadas'].sum(), 1)),
    'runs_no_completados': int((tabla_fallos['estado'] != 'completed').sum()),
    'llamadas_totales': int(tabla_fallos['llamadas'].sum()),
    'contexto_total_caracteres': int(tabla_fallos['caracteres_observacion'].sum()),
}])
display(metricas_fallos)
metricas_fallos.to_csv(raiz_curso / 'outputs' / '09_02_agent_failure_metrics.csv', index=False)

## 10. Discusión

La detección de repetición exacta no detecta bucles semánticos con argumentos distintos; para eso hacen falta presupuesto y criterios de progreso. El historial crece aunque una herramienta no falle. Asegura límites globales de tiempo, llamadas, tokens y permisos, y presenta una terminación explícita.

## 11. Ejercicios

1. Añade un fallo de parseo JSON y mide el estado terminal.
2. Implementa una política que rechace dos llamadas de clima consecutivas.
3. Compara truncar observaciones con resumirlas.
4. Añade un error de herramienta transitorio y define una política de reintento con límite.

## 12. Desafío

Crea una batería adversarial de fallos con criterios de parada y estados observables. Compara el agente sin salvaguardas con versiones que añaden límites, deduplicación y validación, midiendo precisión de tarea, llamadas y tasa de fallo.

## 13. Ideas clave

- Una herramienta válida aún puede ser innecesaria o incorrecta.
- Controla ciclos exactos y semánticos con límites separados.
- El contexto de agente crece con cada observación.
- Registra estado final, errores, llamadas y evidencia de cada paso.